# Model 2 – Deep Interest Network (DIN) Feature Ablation

- Evaluate how different inputs contribute to a DIN-style recommendation model.
- Compare the full model against variants that remove one feature group or mechanism at a time.
- Keep architecture and training settings fixed across variants, using seeds **0, 1 and 2**.

| Methodology | Approach |
|---|---|
| Training | Train on sampled training slates using weighted binary cross-entropy for any interaction; labels 0–3 receive weights `[1,1,5,10]` |
| Model selection | Early-stop each run using val1 purchase NDCG@10 and restore its best checkpoint |
| Evaluation | Use purchase relevance `[0,0,0,1]`; average metrics over searches containing a purchase |
| Comparison | Report mean performance and standard deviation across three training seeds |
| Confidence intervals | Bootstrap searches after averaging paired differences against the full model across seeds |
| Held-out data | Val2 and test are not used |

## 1. Administrivia

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab: set the paths in Settings to local folders.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from copy import deepcopy
from IPython.display import display

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "no GPU: training will be very slow")
print("numpy", np.__version__, "| pandas", pd.__version__, "| pyarrow", pa.__version__)

torch 2.14.1+cu130 | device: cuda | Tesla T4
numpy 2.4.6 | pandas 2.2.3 | pyarrow 23.0.1


In [ ]:
PROJECT = Path("/content/drive/MyDrive/Grp8Submission/pipeline_data")
SEQ_DIR = PROJECT / "step2_model2_seq"        # sequence inputs
SCALAR_DIR = PROJECT / "step2_model2_scalar"  # scalar features
OUT_DIR = PROJECT / "step3_model2"            # this notebook's outputs

SPLITS = ["train", "val1"]
TUNE_SPLIT = "val1"     # early stopping, tuning, ablation
for d in (SEQ_DIR, SCALAR_DIR, OUT_DIR.parent):
    assert d.is_dir(), f"{d} not found: check the shortcut to the shared Grp8Submission folder"
OUT_DIR.mkdir(exist_ok=True)

CFG = dict(
    channels=["click", "cart", "order"],
    use_query_terms=True,          # historical query terms
    use_name_terms=True,           # candidate product-name terms
    use_attributes=True,           # brand / L4 category / shop embeddings
    use_decay=True,                # explicit recency decay in attention
    use_scalars=True,              # Scalar scalar features (Section 1.4.3)
    use_popularity=True,           # enabled by add_popularity

    use_cand_item_id=True,         # retain candidate item-ID embedding
    use_attention=True,            # retain DIN attention
    use_sequence_summaries=True,   # internal summaries; popularity is separate
    use_brand_cate_summaries=True, # affinity ratios and dominant-brand matches
    use_dominant_brand_ids=True,   # legacy name: binary is_dominant_brand features

    # Architecture
    emb_dim=64,
    att_hidden=64,
    mlp_hidden=(256, 128),
    dropout=0.2,
    emb_dropout=0.0,

    # Optimisation
    lr=1e-3,
    weight_decay=0.0,
    emb_weight_decay=0.0,
    grad_clip=5.0,

    # Batching: keep whole searches together
    batch_cands=4096,
    eval_batch_cands=8192,

    # Early stopping on val1 purchase NDCG@10
    max_epochs=8,
    evals_per_epoch=2,
    patience=3,                    # validation checks without improvement

    # Runtime
    amp=True,                      # mixed precision when CUDA is available
    seed=0,
)
GRADED_WEIGHT = {0: 1.0, 1: 1.0, 2: 5.0, 3: 10.0}   # label -> sample weight (interim report, as Model 1)

SEQ_FIELDS = ["item", "brand_id", "cate_id_4", "shop_id", "action", "time_bucket", "has_query", "qry_terms", "days"]
ATTR_COLS = ["brand_id", "cate_id_4", "shop_id"]
SEEDS = [0,1,2]


## 2. Input Loading

- Load sequence and scalar inputs for train and val1.
- Prepare aligned candidate-level inputs for training and evaluation.


### 2.1. Sequence Input Loading

- Load vocabulary information, channel definitions and recency half-lives.
- Reconstruct click, cart and order histories, along with candidate information and labels.
- Associate each candidate with its search’s history.

In [ ]:
VOCAB = json.loads((SEQ_DIR / "vocab.json").read_text())
CHANNELS = {ch: (v["actions"], v["window_days"]) for ch, v in VOCAB["channels"].items()}
HALF_LIFE = {ch: max(v["half_life_days"], 1e-3) for ch, v in VOCAB["channels"].items()}


def to_grid(arr):
    """Arrow (nested) fixed-size list array -> numpy array [n, k] or [n, k, m]."""
    shape, t = [len(arr)], arr.type
    while pa.types.is_fixed_size_list(t):
        shape.append(t.list_size)
        arr, t = arr.flatten(), t.value_type
    return arr.to_numpy(zero_copy_only=False).reshape(shape)


def load_split(name, out_dir=None):
    """(slate_id, user, seqs, cand) exactly as saved by Sequence."""
    out_dir = out_dir or SEQ_DIR
    t = pq.read_table(out_dir / f"seq_{name}.parquet")
    user, seqs, cand = {}, {ch: {} for ch in CHANNELS}, {}
    for col in t.column_names:
        arr = t.column(col).combine_chunks()
        prefix, _, field = col.partition("_")
        if prefix in CHANNELS:
            seqs[prefix][field] = to_grid(arr) if pa.types.is_fixed_size_list(arr.type) \
                                  else arr.to_numpy(zero_copy_only=False)
        elif prefix == "cand":
            values = arr.flatten()
            cand[field] = to_grid(values) if pa.types.is_fixed_size_list(values.type) else values.to_numpy()
            if "row" not in cand:
                counts = arr.value_lengths().to_numpy()
                cand["row"] = np.repeat(np.arange(len(arr)), counts).astype(np.int32)
                cand["pos"] = (np.arange(counts.sum()) - np.repeat(np.cumsum(counts) - counts, counts)).astype(np.int16)
        elif col != "slate_id":
            user[col] = arr.to_numpy()
    return t.column("slate_id").to_numpy(), user, seqs, cand


RAW = {split: load_split(split) for split in SPLITS}
for split, (sid, user, seqs, cand) in RAW.items():
    caps = {ch: s["item"].shape[1] for ch, s in seqs.items()}
    print(f"{split}: {len(sid):,} searches | {len(cand['item']):,} candidates | "
          f"purchases {int((cand['label'] == 3).sum()):,} | channel caps {caps}")

train: 122,083 searches | 2,608,369 candidates | purchases 14,774 | channel caps {'click': 65, 'cart': 80, 'order': 49}
val1: 17,799 searches | 1,589,673 candidates | purchases 2,016 | channel caps {'click': 65, 'cart': 80, 'order': 49}


### 2.2. Scalar Input Loading

- Load scalar features and align searches and candidate counts with the sequence inputs.
- Assemble history summaries and calculate candidate popularity from training histories.
- Features outside [0, 1] are log-transformed (signed `log1p`), every feature is standardised with the train mean and standard deviation, missing values become 0 after standardisation, and a missing-value flag is added for every feature that has missing values in train.
- Select the scalar columns required by each ablation configuration.


In [ ]:
def read_scalar(split, slate_id, cand_counts):
    """Load candidate scalar features, aligned to sequence slate/candidate order."""
    path = SCALAR_DIR / f"scalar_{split}.parquet"

    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found: set SCALAR_DIR, "
            "or CFG['use_scalars'] = False to train without it"
        )

    t = pq.read_table(path)
    sid = t.column("slate_id").to_numpy()
    pos = pd.Series(np.arange(len(sid)), index=sid)

    if pos.index.duplicated().any():
        raise ValueError(f"{split}: duplicate slate_id in Scalar file")

    missing = np.setdiff1d(slate_id, sid)
    if len(missing):
        raise ValueError(
            f"{split}: {len(missing):,} searches missing from Scalar file"
        )

    order = pa.array(pos.loc[slate_id].to_numpy(), type=pa.int64())
    out = {}

    def extract(name, values):
        # A candidate struct contains one field per scalar feature.
        if pa.types.is_struct(values.type):
            for field in values.type:
                child_name = f"{name}__{field.name}" if name else field.name
                # Propagates null parent structs to their child values.
                child = pc.struct_field(values, field.name)
                extract(child_name, child)
            return

        if not (
            pa.types.is_integer(values.type)
            or pa.types.is_floating(values.type)
            or pa.types.is_boolean(values.type)
            or pa.types.is_decimal(values.type)
            or pa.types.is_null(values.type)
        ):
            raise TypeError(
                f"{split}: feature '{name}' has unsupported type "
                f"{values.type}; expected a numeric scalar"
            )

        if name in out:
            raise ValueError(f"{split}: duplicate feature name '{name}'")

        out[name] = (
            pc.cast(values, pa.float64())
            .to_numpy(zero_copy_only=False)
            .astype(np.float64)
        )

    for col in t.column_names:
        if col == "slate_id":
            continue

        arr = pc.take(t.column(col).combine_chunks(), order)

        if not (
            pa.types.is_list(arr.type)
            or pa.types.is_large_list(arr.type)
            or pa.types.is_fixed_size_list(arr.type)
        ):
            raise TypeError(
                f"{split}: column '{col}' has type {arr.type}; "
                "expected one candidate list per slate"
            )

        if arr.null_count:
            raise ValueError(
                f"{split}: column '{col}' contains null candidate lists"
            )

        lens = pc.list_value_length(arr).to_numpy(zero_copy_only=False)
        if not np.array_equal(lens, cand_counts):
            raise ValueError(
                f"{split}: '{col}' has a different number of "
                "candidates than the sequence inputs"
            )

        values = pc.list_flatten(arr)

        if pa.types.is_struct(values.type):
            # Keep the original feature names for channel filtering.
            extract("", values)
        else:
            extract(col, values)

    return out


def item_popularity():
    """Times each known item appears in TRAIN histories, per channel (train-only statistic).
    Unknown (1) and padding (0) get 0. Counts come from the capped channel grids."""
    pop = {}
    for ch, s in RAW["train"][2].items():
        c = np.bincount(s["item"].ravel(), minlength=VOCAB["n_items"]).astype(np.float64)
        c[:2] = 0.0
        pop[ch] = c
    return pop


POPULARITY = item_popularity()


def sequence_summaries(user, seqs, cand):
    """Per-candidate copies of the Sequence per-channel summaries."""
    row = cand["row"]
    # raw counts and days: ScalarTransform applies log1p once (they are outside [0, 1])
    out = {"history_len": user["history_len"].astype(np.float64)[row],
           "cand_has_metadata": cand["has_metadata"].astype(np.float64)}
    for ch, s in seqs.items():
        # out[f"{ch}_count"] = s["full_len"].astype(np.float64)[row]
        out[f"{ch}_days_since_last"] = s["days_since_last"].astype(np.float64)[row]   # NaN if the channel is empty
        out[f"{ch}_empty"] = (s["full_len"] == 0).astype(np.float64)[row]
        out[f"{ch}_popularity"] = POPULARITY[ch][cand["item"]]                    # candidate popularity (raw count)
    return out


def raw_scalars(split):
    """
    Extract raw scalar features
    """
    sid, user, seqs, cand = RAW[split]
    feats = {f"sequence__{k}": v for k, v in sequence_summaries(user, seqs, cand).items()}
    counts = np.bincount(cand["row"], minlength=len(sid))
    if CFG["use_scalars"] or SCALAR_DIR.exists():
        try:
            feats.update({f"scalar__{k}": v for k, v in read_scalar(split, sid, counts).items()})
        except FileNotFoundError as e:
            if CFG["use_scalars"]:
                raise
            print(e)
    return feats


class ScalarTransform:
    """Fitted on train: signed log1p for features outside [0, 1], standardise, NaN -> 0 plus a flag."""
    def fit(self, feats):
        self.names, self.log, self.mean, self.std, self.flag = [], {}, {}, {}, []
        for k, v in feats.items():
            finite = v[np.isfinite(v)]
            self.log[k] = bool(len(finite)) and (finite.min() < 0 or finite.max() > 1)
            x = self._log(k, v)
            xf = x[np.isfinite(x)]
            self.mean[k] = float(xf.mean()) if len(xf) else 0.0
            sd = float(xf.std()) if len(xf) else 0.0
            self.std[k] = sd if sd > 1e-12 else 1.0
            self.names.append(k)
            if np.isnan(v).any():
                self.flag.append(k)
        self.columns = self.names + [f"{k}__missing" for k in self.flag]
        return self

    def _log(self, k, v):
        return np.sign(v) * np.log1p(np.abs(v)) if self.log[k] else v

    def transform(self, feats):
        if set(feats) != set(self.names):
            raise ValueError(f"scalar features differ from train: {set(feats) ^ set(self.names)}")
        cols = []
        for k in self.names:
            x = (self._log(k, feats[k]) - self.mean[k]) / self.std[k]
            cols.append(np.where(np.isfinite(x), x, 0.0))
        cols += [np.isnan(feats[k]).astype(np.float64) for k in self.flag]
        return np.column_stack(cols).astype(np.float32)


raw_sc = {split: raw_scalars(split) for split in RAW}
SCALER = ScalarTransform().fit(raw_sc["train"])
SCALARS = {split: SCALER.transform(f) for split, f in raw_sc.items()}
del raw_sc
SCALAR_NAMES = SCALER.columns
n_scalar = sum(c.startswith("scalar__") and not c.endswith("__missing") for c in SCALAR_NAMES)
print(
    f"{len(SCALAR_NAMES)} scalar inputs: "
    f"{n_scalar} from Scalar, "
    f"{sum(c.startswith('sequence__') and not c.endswith('__missing') for c in SCALAR_NAMES)} internal summaries, "
    f"{len(SCALER.flag)} missing-value flags\n"
)

for i, name in enumerate(SCALAR_NAMES, start=1):
    print(f"{i:2d}. {name}")


def scalar_columns(cfg, names=None):
    """Return the scalar-column indices enabled by this configuration."""
    names = SCALAR_NAMES if names is None else names
    dropped = set(CHANNELS) - set(cfg["channels"])
    keep = []

    for i, name in enumerate(names):
        # Apply the same filtering to a feature and its missingness flag.
        base = name.removesuffix("__missing")
        src, _, feat = base.partition("__")

        # Support both flat and nested upstream feature names.
        local = (
            feat.replace("__", "_")
            .removeprefix("missing_value_flags_")
        )

        channel = next(
            (ch for ch in CHANNELS if local.startswith(f"{ch}_")),
            None,
        )
        feature = local[len(channel) + 1:] if channel else local

        is_popularity = (
            feature == "popularity"
            or feature.endswith("_popularity")
        )
        is_dominant_match = feature == "is_dominant_brand"
        is_brand_cate = feature in {
            "is_dominant_brand",
            "brand_affinity_ratio",
            "cate_affinity_ratio",
        }

        # Raw categorical IDs must not enter as continuous scalars.
        # Binary is_dominant_brand indicators are valid numeric features.
        if "dominant_brand" in feature and not is_dominant_match:
            continue

        # Remove channel-specific features.
        if channel in dropped:
            continue

        # Remove all externally loaded Scalar features.
        if src == "scalar" and not cfg.get("use_scalars", True):
            continue

        # Popularity is controlled separately from sequence summaries.
        if is_popularity:
            if not cfg.get("use_popularity", True):
                continue
        elif (
            src == "sequence"
            and not cfg.get("use_sequence_summaries", True)
        ):
            continue

        # Remove brand/category summaries, retaining item-history counts.
        if (
            src == "scalar"
            and is_brand_cate
            and not cfg.get("use_brand_cate_summaries", True)
        ):
            continue

        # Legacy flag name: controls binary dominant-brand matches.
        if (
            is_dominant_match
            and not cfg.get("use_dominant_brand_ids", True)
        ):
            continue

        keep.append(i)

    return np.asarray(keep, dtype=np.int64)

30 scalar inputs: 16 from Scalar, 11 internal summaries, 3 missing-value flags

 1. sequence__history_len
 2. sequence__cand_has_metadata
 3. sequence__click_days_since_last
 4. sequence__click_empty
 5. sequence__click_popularity
 6. sequence__cart_days_since_last
 7. sequence__cart_empty
 8. sequence__cart_popularity
 9. sequence__order_days_since_last
10. sequence__order_empty
11. sequence__order_popularity
12. scalar__is_dominant_brand
13. scalar__brand_affinity_ratio
14. scalar__cate_affinity_ratio
15. scalar__hist_item_count
16. scalar__click_is_dominant_brand
17. scalar__click_brand_affinity_ratio
18. scalar__click_cate_affinity_ratio
19. scalar__click_hist_item_count
20. scalar__cart_is_dominant_brand
21. scalar__cart_brand_affinity_ratio
22. scalar__cart_cate_affinity_ratio
23. scalar__cart_hist_item_count
24. scalar__order_is_dominant_brand
25. scalar__order_brand_affinity_ratio
26. scalar__order_cate_affinity_ratio
27. scalar__order_hist_item_count
28. sequence__click_days_s

## 3. Batching

- Group candidates into batches while keeping each search's candidate slate intact.
- Share each search's history across its candidates and gather the selected scalar inputs.
- Convert batch arrays into tensors on the selected device.

In [ ]:
class SplitData:
    def __init__(self, split):
        self.split = split
        self.slate_id, self.user, self.seqs, self.cand = RAW[split]
        self.scalars = SCALARS[split]
        n = len(self.slate_id)
        self.counts = np.bincount(self.cand["row"], minlength=n)
        self.starts = np.concatenate([[0], np.cumsum(self.counts)[:-1]])
        self.has_purchase = np.bincount(self.cand["row"], weights=(self.cand["label"] == 3), minlength=n) > 0
        assert np.all(np.diff(self.cand["row"]) >= 0), "candidates must be grouped by search"

    def batches(self, max_cands, shuffle=False, rng=None, slates=None):
        idx = np.arange(len(self.slate_id)) if slates is None else np.asarray(slates)
        if shuffle:
            idx = rng.permutation(idx)
        out, cur, n = [], [], 0
        for s in idx:
            c = self.counts[s]
            if cur and n + c > max_cands:
                out.append(np.array(cur)); cur, n = [], 0
            cur.append(s); n += c
        if cur:
            out.append(np.array(cur))
        return out

    def gather(self, sl, channels, scalar_cols):
        """Numpy arrays for one batch of searches `sl`."""
        lens = self.counts[sl]
        cidx = np.repeat(self.starts[sl], lens) + (np.arange(lens.sum()) - np.repeat(np.cumsum(lens) - lens, lens))
        return {
            "seq": {ch: {f: self.seqs[ch][f][sl] for f in SEQ_FIELDS} for ch in channels},
            "cand": {f: self.cand[f][cidx] for f in ["item", *ATTR_COLS, "name_terms", "label"]},
            "row": np.repeat(np.arange(len(sl)), lens),
            "scalars": self.scalars[cidx][:, scalar_cols],
            "cidx": cidx,
        }


DATA = {split: SplitData(split) for split in RAW}
for split, d in DATA.items():
    print(f"{split}: {d.has_purchase.sum():,} searches with a purchase "
          f"(of {len(d.slate_id):,}); median {int(np.median(d.counts))} candidates per search")

train: 14,408 searches with a purchase (of 122,083); median 21 candidates per search
val1: 1,945 searches with a purchase (of 17,799); median 90 candidates per search


In [ ]:
def to_device(b):
    def t(x, float_=False):
        x = torch.from_numpy(np.ascontiguousarray(x))
        return (x.float() if float_ else x.long()).to(DEVICE, non_blocking=True)
    return {
        "seq": {ch: {f: t(v, float_=(f == "days")) for f, v in s.items()} for ch, s in b["seq"].items()},
        "cand": {f: t(v) for f, v in b["cand"].items()},
        "row": t(b["row"]),
        "scalars": t(b["scalars"], float_=True),
    }

## 4. DIN network
- Embed product identifiers, attributes, historical queries and candidate names.
- Build historical event representations and apply candidate-conditioned attention within each channel to create concatenate channel interests.
- Combine channel summaries, the candidate representation and scalar features to produce a ranking logit.
- Define the weighted binary classification loss used for training.

In [ ]:
class Model2(nn.Module):
    def __init__(self, cfg, n_scalar):
        super().__init__()
        d, h = cfg["emb_dim"], cfg["att_hidden"]
        self.cfg = cfg
        self.item = nn.Embedding(VOCAB["n_items"], d, padding_idx=0)
        self.attr = nn.ModuleDict({c: nn.Embedding(VOCAB[f"n_{c}"], d, padding_idx=0) for c in ATTR_COLS})
        self.term = nn.Embedding(VOCAB["n_terms"], d, padding_idx=0)
        self.action = nn.Embedding(VOCAB["n_actions"], d, padding_idx=0)
        self.tbucket = nn.Embedding(VOCAB["n_time_buckets"], d, padding_idx=0)
        self.emb_drop = nn.Dropout(cfg.get("emb_dropout", 0.0))
        self.event = nn.Linear(4 * d + 1, d)
        self.cand = nn.Linear(2 * d, d)
        self.att = nn.ModuleDict({ch: nn.Sequential(nn.Linear(4 * d, h), nn.ReLU(), nn.Linear(h, 1))
                                  for ch in cfg["channels"]})
        layers, width = [], len(cfg["channels"]) * d + d + n_scalar
        for out in cfg["mlp_hidden"]:
            layers += [nn.Linear(width, out), nn.ReLU(), nn.Dropout(cfg["dropout"])]
            width = out
        self.mlp = nn.Sequential(*layers, nn.Linear(width, 1))
        self.embedding_tables = [self.item, *self.attr.values(), self.term]
        for emb in [self.item, *self.attr.values(), self.term, self.action, self.tbucket]:
            nn.init.normal_(emb.weight, std=0.05)
            with torch.no_grad():
                emb.weight[0].zero_()

    def product(self, x, use_item=True):
        v = self.item(x["item"]) if use_item else 0.0
        if self.cfg["use_attributes"]:
            for c in ATTR_COLS:
                v = v + self.attr[c](x[c])
        if not torch.is_tensor(v):          # neither item ID nor attributes: a zero vector
            v = torch.zeros(*x["item"].shape, self.item.embedding_dim, device=x["item"].device)
        return self.emb_drop(v)

    def words(self, terms):
        e = self.emb_drop(self.term(terms))
        m = (terms > 0).unsqueeze(-1).to(e.dtype)
        return (e * m).sum(-2) / m.sum(-2).clamp(min=1.0)

    def forward(self, b, return_attn=False):
        c = b["cand"]
        name = self.words(c["name_terms"]) if self.cfg["use_name_terms"] else \
               torch.zeros(len(c["item"]), self.item.embedding_dim, device=c["item"].device)
        cvec = self.cand(torch.cat([self.product(c, use_item=self.cfg.get("use_cand_item_id", True)), name], -1))  # [N, d]
        row, parts, attn = b["row"], [], {}
        for ch in self.cfg["channels"]:
            s = b["seq"][ch]
            if self.cfg["use_query_terms"]:
                q = self.words(s["qry_terms"])
            else:
                q = torch.zeros(*s["item"].shape, self.item.embedding_dim, device=s["item"].device)
            ev = self.event(torch.cat([self.product(s), self.action(s["action"]), self.tbucket(s["time_bucket"]),
                                       q, (s["has_query"] if self.cfg["use_query_terms"] else torch.zeros_like(s["has_query"])).unsqueeze(-1).float()], -1))  # [B, L, d]
            H = ev[row]                                                               # [N, L, d]
            C = cvec.unsqueeze(1).expand_as(H)
            logits = self.att[ch](torch.cat([H, C, H - C, H * C], -1)).squeeze(-1).float()
            if self.cfg["use_decay"]:
                logits = logits - math.log(2) * s["days"][row] / HALF_LIFE[ch]
            mask = s["action"][row] > 0
            w = torch.softmax(logits.masked_fill(~mask, -1e9), dim=-1) * mask.float()  # empty channel -> all 0
            if not self.cfg.get("use_attention", True):
                w = mask.float() / mask.sum(-1, keepdim=True).clamp(min=1)
            parts.append((w.unsqueeze(-1) * H.float()).sum(1))
            if return_attn:
                attn[ch] = w.detach()
        x = torch.cat(parts + [cvec.float(), b["scalars"]], -1)
        out = self.mlp(x).squeeze(-1)
        return (out, attn) if return_attn else out


WEIGHT_T = torch.tensor([GRADED_WEIGHT[k] for k in range(4)], device=DEVICE)


def weighted_bce(logit, label):
    """Graded objective: target = any interaction, weight = 1 / 1 / 5 / 10 for labels 0 / 1 / 2 / 3."""
    w = WEIGHT_T[label]
    y = (label >= 1).float()
    loss = (F.binary_cross_entropy_with_logits(logit.float(), y, reduction="none") * w).sum() / w.sum()
    return loss, float(w.sum())

## 5. Metrics

- Calculate purchase-based NDCG@10, Hit@10, Hit@1 and MRR for each eligible search (Code referenced from Model 1 for consistency).
- Average results over five seeded tie-breaks and exclude searches without purchases.
- Provide prediction, evaluation and paired-comparison utilities.

In [ ]:
METRICS = ["ndcg@10", "hit@10", "hit@1", "mrr"]


def _gains(label, relevance):
    label = np.asarray(label)
    if relevance == "purchase":
        return (label == 3).astype(np.float64)
    if relevance == "graded":
        return (2.0 ** label - 1.0).astype(np.float64)
    raise ValueError("relevance must be 'purchase' or 'graded'")


def _group_layout(sorted_slates):
    n = len(sorted_slates)
    start = np.r_[0, np.flatnonzero(sorted_slates[1:] != sorted_slates[:-1]) + 1]
    size = np.diff(np.r_[start, n])
    rank = np.arange(n) - np.repeat(start, size) + 1
    return start, rank


def per_slate_metrics(df, score_col, k=10, relevance="purchase", higher_is_better=True,
                      n_tiebreaks=5, seed=0):
    slate = df["slate_id"].to_numpy()
    gain = _gains(df["label"].to_numpy(), relevance)
    has_rel = pd.Series(gain > 0).groupby(slate).transform("any").to_numpy()
    slate, gain = slate[has_rel], gain[has_rel]
    score = df[score_col].to_numpy(dtype=np.float64)[has_rel]
    if len(slate) == 0:
        raise ValueError("no slates with a relevant item")
    key = score if higher_is_better else -score
    key = np.where(np.isnan(key), -np.inf, key)
    ideal_order = np.lexsort((-gain, slate))
    start, rank = _group_layout(slate[ideal_order])
    disc = np.where(rank <= k, 1.0 / np.log2(rank + 1), 0.0)
    idcg = np.add.reduceat(gain[ideal_order] * disc, start)
    rng = np.random.default_rng(seed)
    acc = {m: np.zeros(len(start)) for m in METRICS}
    for _ in range(n_tiebreaks):
        tie = rng.random(len(slate))
        order = np.lexsort((tie, -key, slate))
        g = gain[order]
        rel = g > 0
        acc["ndcg@10"] += np.add.reduceat(g * disc, start) / idcg
        acc["hit@10"] += np.add.reduceat((rel & (rank <= k)).astype(float), start) > 0
        acc["hit@1"] += np.add.reduceat((rel & (rank == 1)).astype(float), start) > 0
        first = np.minimum.reduceat(np.where(rel, rank, np.inf), start)
        acc["mrr"] += 1.0 / first
    out = pd.DataFrame({m: v / n_tiebreaks for m, v in acc.items()})
    out = out.rename(columns={"ndcg@10": f"ndcg@{k}", "hit@10": f"hit@{k}"})
    out.insert(0, "slate_id", slate[ideal_order][start])
    return out


def paired_diff(a, b, col="ndcg@10", n_boot=2000, seed=0):
    # mean of a - b over slates, with a 95% bootstrap interval (a and b: per-slate tables on the same slates)
    assert (a["slate_id"].to_numpy() == b["slate_id"].to_numpy()).all(), "slates not aligned"
    d = a[col].to_numpy() - b[col].to_numpy()
    rng = np.random.default_rng(seed)
    boot = np.concatenate([d[rng.integers(0,len(d),size=(min(100,n_boot-i),len(d)))].mean(axis=1)
                           for i in range(0,n_boot,100)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {"diff": d.mean(), "ci_low": lo, "ci_high": hi}


def candidate_frame(data, scores, slates=None):
    """slate_id / position / label / score for the candidates of `slates` (default: all)."""
    rows = np.arange(len(data.slate_id)) if slates is None else np.asarray(slates)
    lens = data.counts[rows]
    cidx = np.repeat(data.starts[rows], lens) + (np.arange(lens.sum()) - np.repeat(np.cumsum(lens) - lens, lens))
    return pd.DataFrame({"slate_id": data.slate_id[data.cand["row"][cidx]], "position": data.cand["pos"][cidx],
                         "label": data.cand["label"][cidx], "score": scores[cidx]})

In [ ]:
@torch.no_grad()
def predict(model, data, slates=None):
    """Raw logits for every candidate of `slates` (others NaN), in the data's candidate order."""
    model.eval()
    cols = scalar_columns(model.cfg)
    scores = np.full(len(data.cand["item"]), np.nan, dtype=np.float64)
    for sl in data.batches(model.cfg["eval_batch_cands"], slates=slates):
        b = data.gather(sl, model.cfg["channels"], cols)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=model.cfg["amp"] and DEVICE.type == "cuda"):
            out = model(to_device(b))
        scores[b["cidx"]] = out.float().cpu().numpy()
    return scores


def evaluate(model, data, slates=None):
    """Per-search metrics on searches with a purchase."""
    slates = np.flatnonzero(data.has_purchase) if slates is None else slates
    return per_slate_metrics(candidate_frame(data, predict(model, data, slates), slates), "score")

## 6. Training

- Initialise each run using its assigned seed and train with AdamW, gradient clipping and optional mixed precision.
- Check val1 performance during training and stop after repeated checks without improvement.
- Restore the checkpoint with the highest val1 purchase NDCG@10.

In [ ]:
def make_optimizer(model, cfg):
    """AdamW. Embedding tables (items, brands, categories, shops, words) get `emb_weight_decay`;
    everything else gets `weight_decay`. With both at 0 this is plain Adam."""
    emb_ids = {id(p) for t in model.embedding_tables for p in t.parameters()}
    emb = [p for p in model.parameters() if id(p) in emb_ids]
    rest = [p for p in model.parameters() if id(p) not in emb_ids]
    return torch.optim.AdamW([{"params": emb, "weight_decay": cfg.get("emb_weight_decay", 0.0)},
                              {"params": rest, "weight_decay": cfg["weight_decay"]}], lr=cfg["lr"])


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def free_gpu():
    """Release GPU memory from models that are no longer referenced."""
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()




In [ ]:
def make_optimizer(model, cfg):
    """AdamW. Embedding tables (items, brands, categories, shops, words) get `emb_weight_decay`;
    everything else gets `weight_decay`. With both at 0 this is plain Adam."""
    emb_ids = {id(p) for t in model.embedding_tables for p in t.parameters()}
    emb = [p for p in model.parameters() if id(p) in emb_ids]
    rest = [p for p in model.parameters() if id(p) not in emb_ids]
    return torch.optim.AdamW([{"params": emb, "weight_decay": cfg.get("emb_weight_decay", 0.0)},
                              {"params": rest, "weight_decay": cfg["weight_decay"]}], lr=cfg["lr"])


def train_model(cfg, fixed_points=None, verbose=True):
    """Train one Model 2, checking val1 `evals_per_epoch` times per epoch.
    fixed_points=None: early stopping on val1 NDCG@10 (keeps the best check's weights);
    otherwise train exactly that many check intervals (used by the ablation)."""
    set_seed(cfg["seed"])
    cols = scalar_columns(cfg)
    model = Model2(cfg, len(cols)).to(DEVICE)
    opt = make_optimizer(model, cfg)
    use_amp = cfg["amp"] and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    rng = np.random.default_rng(cfg["seed"])
    tr, va = DATA["train"], DATA[TUNE_SPLIT]
    k = max(1, int(cfg.get("evals_per_epoch", 1)))
    history, best, best_state, best_point, bad, point = [], -1.0, None, 0, 0, 0
    t0, total, wsum = time.time(), 0.0, 0.0
    for epoch in range(1, cfg["max_epochs"] + 1):
        batches = tr.batches(cfg["batch_cands"], shuffle=True, rng=rng)
        parts = np.array_split(np.arange(len(batches)), k)
        for part in parts:
            if len(part) == 0:
                continue
            model.train()
            for bi in part:
                b = to_device(tr.gather(batches[bi], cfg["channels"], cols))
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    out = model(b)
                loss, w = weighted_bce(out, b["cand"]["label"])
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
                scaler.step(opt)
                scaler.update()
                total += loss.item() * w; wsum += w
            point += 1
            val = evaluate(model, va)
            rec = {"check": point, "epoch": round(point / k, 2), "train_loss": total / max(wsum, 1e-12),
                   **{f"{TUNE_SPLIT}_{m}": val[m].mean() for m in METRICS}, "minutes": (time.time() - t0) / 60}
            history.append(rec)
            t0, total, wsum = time.time(), 0.0, 0.0
            if verbose:
                print(f"epoch {rec['epoch']:.2f}: train loss {rec['train_loss']:.4f} | "
                      f"{TUNE_SPLIT} NDCG@10 {rec[TUNE_SPLIT + '_ndcg@10']:.4f} | "
                      f"{TUNE_SPLIT} Hit@10 {rec[TUNE_SPLIT + '_hit@10']:.4f} | {rec['minutes']:.1f} min")
            if fixed_points is not None:
                if point >= fixed_points:
                    return model, pd.DataFrame(history), point
                continue
            if rec[f"{TUNE_SPLIT}_ndcg@10"] > best:
                best, best_point, bad = rec[f"{TUNE_SPLIT}_ndcg@10"], point, 0
                best_state = {kk: v.detach().cpu().clone() for kk, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= cfg["patience"]:
                    model.load_state_dict(best_state)
                    return model, pd.DataFrame(history), best_point
    if fixed_points is None:
        model.load_state_dict(best_state)
        return model, pd.DataFrame(history), best_point
    return model, pd.DataFrame(history), point




## 7. Ablation Runs and Results

- Define the full configuration and variants that remove individual inputs or mechanisms.
- Display each variant’s selected scalar features and run all configurations across three seeds.
- Retain aggregate and per-search results in memory for comparison.
- Summarise average performance, seed variability and bootstrap confidence intervals, then identify the best-performing configuration.

In [ ]:
ABLATIONS = {
    "full": {},
    "no_click": {"channels": ["cart", "order"]},
    "no_cart": {"channels": ["click", "order"]},
    "no_order": {"channels": ["click", "cart"]},
    "no_history_query": {"use_query_terms": False},
    "no_candidate_name": {"use_name_terms": False},
    "no_attributes": {"use_attributes": False},
    "no_recency_decay": {"use_decay": False},
    "no_scalar_scalars": {"use_scalars": False},
    "no_popularity": {"use_popularity": False},
    "no_cand_item_id": {"use_cand_item_id": False},
    "no_sequence_summaries": {"use_sequence_summaries": False},
    "no_brand_cate_summaries": {"use_brand_cate_summaries": False},
    "no_dominant_brand_ids": {"use_dominant_brand_ids": False},
}

In [ ]:
for name, overrides in ABLATIONS.items():
    cfg = {**CFG, **overrides}
    cols = scalar_columns(cfg)

    print(f"\n{name}: {len(cols)} scalar inputs")
    for i in cols:
        print(f"  {SCALAR_NAMES[i]}")


full: 30 scalar inputs
  sequence__history_len
  sequence__cand_has_metadata
  sequence__click_days_since_last
  sequence__click_empty
  sequence__click_popularity
  sequence__cart_days_since_last
  sequence__cart_empty
  sequence__cart_popularity
  sequence__order_days_since_last
  sequence__order_empty
  sequence__order_popularity
  scalar__is_dominant_brand
  scalar__brand_affinity_ratio
  scalar__cate_affinity_ratio
  scalar__hist_item_count
  scalar__click_is_dominant_brand
  scalar__click_brand_affinity_ratio
  scalar__click_cate_affinity_ratio
  scalar__click_hist_item_count
  scalar__cart_is_dominant_brand
  scalar__cart_brand_affinity_ratio
  scalar__cart_cate_affinity_ratio
  scalar__cart_hist_item_count
  scalar__order_is_dominant_brand
  scalar__order_brand_affinity_ratio
  scalar__order_cate_affinity_ratio
  scalar__order_hist_item_count
  sequence__click_days_since_last__missing
  sequence__cart_days_since_last__missing
  sequence__order_days_since_last__missing

no_clic

In [ ]:
RUN_VARIANTS = list(ABLATIONS)

rows = []
manifest = {}
per_slate_runs = {}
for seed in SEEDS:
    reference = None

    for name in ["full"] + [n for n in RUN_VARIANTS if n != "full"]:
        cfg = deepcopy(CFG)
        cfg.update(deepcopy(ABLATIONS[name]))
        cfg["seed"] = seed

        tag = f"{name}_seed{seed}"
        cols = scalar_columns(cfg)
        manifest[tag] = [SCALAR_NAMES[i] for i in cols]

        print(f"\n{tag}: {len(cols)} scalar inputs")
        free_gpu()
        started = time.time()

        model, history, best_point = train_model(cfg)

        ps = evaluate(model, DATA["val1"]).sort_values("slate_id").reset_index(drop=True)

        if name == "full": reference = ps.copy()
        per_slate_runs[(name, seed)] = ps[["slate_id", *METRICS]].copy()
        row = dict(
            variant=name,
            seed=seed,
            best_check=best_point,
            n_scalars=len(cols),
            n_eval_slates=len(ps),
            minutes=(time.time() - started) / 60,
            **{
                f"val1_{metric}": float(ps[metric].mean())
                for metric in METRICS
            },
        )

        row["delta_vs_full"] = float((
            ps["ndcg@10"].to_numpy()
            - reference["ndcg@10"].to_numpy()
        ).mean())

        rows.append(row)
        print(row)

        del model, history, ps
        free_gpu()


full_seed0: 30 scalar inputs
epoch 0.50: train loss 0.4209 | val1 NDCG@10 0.3605 | val1 Hit@10 0.6015 | 0.9 min
epoch 1.00: train loss 0.4055 | val1 NDCG@10 0.3651 | val1 Hit@10 0.6149 | 0.9 min
epoch 1.50: train loss 0.3213 | val1 NDCG@10 0.3122 | val1 Hit@10 0.5368 | 0.9 min
epoch 2.00: train loss 0.3265 | val1 NDCG@10 0.3209 | val1 Hit@10 0.5512 | 0.9 min
epoch 2.50: train loss 0.2296 | val1 NDCG@10 0.2595 | val1 Hit@10 0.4817 | 0.9 min
{'variant': 'full', 'seed': 0, 'best_check': 2, 'n_scalars': 30, 'n_eval_slates': 1945, 'minutes': 4.564813876152039, 'val1_ndcg@10': 0.36505371908593787, 'val1_hit@10': 0.6149100257069409, 'val1_hit@1': 0.17326478149100258, 'val1_mrr': 0.30896128268551765, 'delta_vs_full': 0.0}

no_click_seed0: 22 scalar inputs
epoch 0.50: train loss 0.4251 | val1 NDCG@10 0.3543 | val1 Hit@10 0.5995 | 0.7 min
epoch 1.00: train loss 0.4080 | val1 NDCG@10 0.3590 | val1 Hit@10 0.6034 | 0.7 min
epoch 1.50: train loss 0.3230 | val1 NDCG@10 0.3122 | val1 Hit@10 0.5404 | 

In [ ]:
def bootstrap_search_ci(values, n_boot=2000, seed=0):
    """95% bootstrap CI over seed-averaged per-search differences."""
    values = np.asarray(values, dtype=float)

    if (
        values.ndim != 1
        or values.size == 0
        or not np.isfinite(values).all()
    ):
        raise ValueError("Expected nonempty, finite 1D differences")

    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot)

    # Batch resampling to limit memory use.
    batch_size = max(1, min(100, 1_000_000 // len(values)))

    for start in range(0, n_boot, batch_size):
        size = min(batch_size, n_boot - start)
        indices = rng.integers(
            0, len(values), size=(size, len(values))
        )
        boot[start:start + size] = values[indices].mean(axis=1)

    lo, hi = np.percentile(boot, [2.5, 97.5])
    return float(values.mean()), float(lo), float(hi)


runs = pd.DataFrame(rows)

summary = runs.groupby("variant", sort=False).agg(
    val1_ndcg10_mean=("val1_ndcg@10", "mean"),
    val1_ndcg10_std=("val1_ndcg@10", "std"),
    n_seeds=("seed", "nunique"),
    minutes_total=("minutes", "sum"),
)

ci_rows = []

for variant in summary.index:
    paired_differences = []
    reference_ids = None

    for seed in SEEDS:
        a = (
            per_slate_runs[(variant, seed)]
            .set_index("slate_id")["ndcg@10"]
            .sort_index()
        )
        b = (
            per_slate_runs[("full", seed)]
            .set_index("slate_id")["ndcg@10"]
            .sort_index()
        )

        if not a.index.is_unique or not b.index.is_unique:
            raise ValueError("Duplicate slate IDs")

        if reference_ids is None:
            reference_ids = b.index

        if (
            not a.index.equals(reference_ids)
            or not b.index.equals(reference_ids)
        ):
            raise ValueError(
                f"{variant}, seed {seed}: searches are not aligned"
            )

        paired_differences.append(
            a.to_numpy(dtype=float) - b.to_numpy(dtype=float)
        )

    # [seeds, searches] -> one difference per search.
    mean_diff_per_search = np.stack(
        paired_differences
    ).mean(axis=0)

    diff, lo, hi = bootstrap_search_ci(mean_diff_per_search)

    ci_rows.append({
        "variant": variant,
        "delta_vs_full_mean": diff,
        "ci_low": lo,
        "ci_high": hi,
        "ci_excludes_0": "*" if lo > 0 or hi < 0 else "",
    })

summary = (
    summary
    .join(pd.DataFrame(ci_rows).set_index("variant"))
    .sort_values("val1_ndcg10_mean", ascending=False)
)

display(summary.round(6))

winner = str(summary.index[0])
selection = {
    "variant": winner,
    "overrides": ABLATIONS[winner],
    "seeds": SEEDS,
    "selection_metric": "val1 purchase NDCG@10",
    "val1_ndcg10_mean": float(summary.iloc[0].val1_ndcg10_mean),
}

print("Selected configuration:", selection)
print(
    "CI: 95% paired bootstrap across searches, "
    "after averaging paired differences across seeds."
)
print("* indicates that the CI excludes zero.")
print("With one seed, seed SD is NaN; search-level CI is available.")
print("This CI does not capture training-seed uncertainty.")
print("Val2/test have not been loaded.")

,val1_ndcg10_mean,val1_ndcg10_std,n_seeds,minutes_total,delta_vs_full_mean,ci_low,ci_high,ci_excludes_0
variant,,,,,,,,
no_cand_item_id,0.365483,0.001190,3,11.319763,0.000957,-0.003431,0.005232,
full,0.364526,0.000641,3,13.687813,0.000000,0.000000,0.000000,
no_history_query,0.364432,0.001446,3,12.827732,-0.000094,-0.001287,0.001229,
no_dominant_brand_ids,0.364368,0.000401,3,13.682162,-0.000158,-0.002304,0.002058,
no_recency_decay,0.364162,0.000785,3,13.688155,-0.000364,-0.001511,0.000768,
no_sequence_summaries,0.364131,0.001900,3,13.666571,-0.000395,-0.002514,0.001858,
no_candidate_name,0.363981,0.000153,3,13.360350,-0.000545,-0.003650,0.002671,
no_order,0.362404,0.001250,3,11.603930,-0.002121,-0.005321,0.001416,
no_cart,0.362209,0.001001,3,10.863197,-0.002317,-0.005713,0.001052,


Selected configuration: {'variant': 'no_cand_item_id', 'overrides': {'use_cand_item_id': False}, 'seeds': [0, 1, 2], 'selection_metric': 'val1 purchase NDCG@10', 'val1_ndcg10_mean': 0.3654832203701608}
CI: 95% paired bootstrap across searches, after averaging paired differences across seeds.
* indicates that the CI excludes zero.
With one seed, seed SD is NaN; search-level CI is available.
This CI does not capture training-seed uncertainty.
Val2/test have not been loaded.


## Interpretation

- A negative difference from `full` indicates that removing the component reduced val1 NDCG@10.
- A confidence interval excluding zero indicates evidence of a difference under search-level bootstrap resampling.
- These intervals do not capture training-seed uncertainty or account for model selection.
- Freeze the chosen configuration before subsequent val2 calibration or final test evaluation.